# 05 - Run Fire and Smoke Detection on Jetson Camera

Use this notebook on the Jetson Nano after copying the trained model. Start with `best.pt`; optimize later if needed.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / 'runs' / 'fire_smoke_yolov8n' / 'weights' / 'best.pt'
ALERT_SOUND = PROJECT_ROOT / 'alert_sound.mp3'

print('Model:', MODEL_PATH)
print('Model exists:', MODEL_PATH.exists())
print('Alert sound exists:', ALERT_SOUND.exists())

In [ ]:
from ultralytics import YOLO
import cv2
import time

model = YOLO(str(MODEL_PATH))

camera_index = 0
cap = cv2.VideoCapture(camera_index)

if not cap.isOpened():
    raise RuntimeError(f'Could not open camera index {camera_index}')

fire_seconds = 0.0
smoke_seconds = 0.0
last_time = time.time()

fire_threshold_conf = 0.55
smoke_threshold_conf = 0.75
fire_duration_threshold = 2.0
smoke_duration_threshold = 3.0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    now = time.time()
    dt = now - last_time
    last_time = now

    results = model.predict(frame, imgsz=640, conf=0.25, verbose=False)
    boxes = results[0].boxes

    fire_detected = False
    smoke_detected = False
    kept_boxes = []

    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        if cls == 0 and conf >= fire_threshold_conf:
            fire_detected = True
            kept_boxes.append(box)
        elif cls == 1 and conf >= smoke_threshold_conf:
            smoke_detected = True
            kept_boxes.append(box)

    fire_seconds = fire_seconds + dt if fire_detected else 0.0
    smoke_seconds = smoke_seconds + dt if smoke_detected else 0.0

    annotated = frame.copy()
    for box in kept_boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        label = f'{model.names[cls]} {conf:.2f}'
        color = (0, 0, 255) if cls == 0 else (180, 180, 180)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    if fire_seconds >= fire_duration_threshold:
        cv2.putText(annotated, 'FIRE ALERT', (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    if smoke_seconds >= smoke_duration_threshold:
        cv2.putText(annotated, 'SMOKE ALERT', (30, 95), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    cv2.imshow('Fire and Smoke Detection', annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

If the camera does not open, try changing `camera_index` from `0` to `1`. For CSI cameras on Jetson Nano, a GStreamer camera pipeline may be needed instead of a plain camera index.